In [ ]:
# Feature selection based on EDA

from IPython.display import display

target_col = "target_churn_from_dac"

# Базово исключаем служебные и явно дублирующие колонки
service_cols = {
    "contact_id",
    target_col,
    "is_dac_next_month",
    "outlier",
}

available_features = [f for f in features if f in df.columns and f not in service_cols]

# Если ранее в ноутбуке уже считались corr_target_report / phik_target, используем их.
# Иначе fallback: считаем Spearman correlation.
if "corr_target_report" in globals():
    feature_rank = corr_target_report.copy()
    feature_rank["score_abs"] = feature_rank[["pearson", "spearman", "phik"]].abs().max(axis=1)
    feature_rank = feature_rank.sort_values("score_abs", ascending=False)
elif "phik_target" in globals():
    feature_rank = phik_target.to_frame("phik")
    feature_rank["score_abs"] = feature_rank["phik"].abs()
    feature_rank = feature_rank.sort_values("score_abs", ascending=False)
else:
    corr = (
        df[available_features + [target_col]]
        .corr(method="spearman", numeric_only=True)[target_col]
        .drop(target_col)
        .abs()
        .sort_values(ascending=False)
    )
    feature_rank = corr.to_frame("spearman_abs")
    feature_rank["score_abs"] = feature_rank["spearman_abs"]

# Missing / uniqueness diagnostics
feature_quality = pd.DataFrame(index=available_features)
feature_quality["missing_share"] = df[available_features].isna().mean()
feature_quality["nunique"] = df[available_features].nunique(dropna=True)

feature_selection_report = (
    feature_rank
    .join(feature_quality, how="left")
    .sort_values("score_abs", ascending=False)
)

# Убираем совсем слабые/плохие кандидаты
selected_features = feature_selection_report[
    (feature_selection_report["missing_share"] < 0.80)
    & (feature_selection_report["nunique"] > 1)
].index.tolist()

# Убираем очевидные дубли DAC count/share: share = count / 12
# Оставляем count, потому что он проще интерпретируется.
drop_manual = {
    "dac_share_last_12",
}

selected_features = [f for f in selected_features if f not in drop_manual]

# Ограничим верхний набор, чтобы первая модель была компактнее.
# Можно поменять TOP_N на 50/80/120.
TOP_N = 80
selected_features = selected_features[:TOP_N]

print("Available features:", len(available_features))
print("Selected features:", len(selected_features))

display(feature_selection_report.head(40))
display(pd.Series(selected_features, name="selected_features").to_frame())

# Проверки
assert len(selected_features) > 0
assert target_col not in selected_features
assert "contact_id" not in selected_features

# Опционально: сохранить список фичей локально
pd.Series(selected_features, name="feature").to_csv(
    "selected_features_churn_from_dac.csv",
    index=False,
)

In [ ]:
# Train/test split and parquet save

from sklearn.model_selection import train_test_split
from pathlib import Path

target_col = "target_churn_from_dac"

# На всякий случай приводим contact_id к int
df["contact_id"] = df["contact_id"].astype("int64")

# Для стратификации лучше учитывать target + DAC-сегмент, если сегмент есть.
if "dac_segment_12m" in df.columns:
    strat_col = df[target_col].astype(str) + "_" + df["dac_segment_12m"].astype(str)
    
    # Если есть слишком маленькие страты, fallback только на target
    strat_counts = strat_col.value_counts()
    if (strat_counts < 2).any():
        stratify = df[target_col]
    else:
        stratify = strat_col
else:
    stratify = df[target_col]

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE if "RANDOM_STATE" in globals() else 42,
    stratify=stratify,
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTarget distribution train:")
display(train_df[target_col].value_counts(normalize=True).to_frame("share"))

print("\nTarget distribution test:")
display(test_df[target_col].value_counts(normalize=True).to_frame("share"))

# Проверка пересечения клиентов
overlap = set(train_df["contact_id"]) & set(test_df["contact_id"])
print("Contact overlap:", len(overlap))
assert len(overlap) == 0, "Есть пересечение contact_id между train и test"

# Проверка selected_features
missing_selected = sorted(set(selected_features) - set(df.columns))
assert len(missing_selected) == 0, f"Нет selected_features в df: {missing_selected}"

# Сохраняем полный набор колонок, но рядом сохраняем список selected_features.
# Так будет удобно переиспользовать датасеты для разных экспериментов.
output_dir = Path("train_test_churn_from_dac")
output_dir.mkdir(parents=True, exist_ok=True)

train_path = output_dir / "train.parquet"
test_path = output_dir / "test.parquet"
features_path = output_dir / "selected_features.parquet"

train_df.to_parquet(train_path, index=False)
test_df.to_parquet(test_path, index=False)
pd.DataFrame({"feature": selected_features}).to_parquet(features_path, index=False)

print("Saved:")
print(train_path.resolve())
print(test_path.resolve())
print(features_path.resolve())